# Common

In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '2,3,4,5'

import numpy as np
from tqdm import tqdm
import pickle as pkl
import torch
from torch.utils.data import TensorDataset, DataLoader
import captum
from pathlib import Path


/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Text
* LayerIntegratedGradients

In [2]:
from transformers import AutoTokenizer
from captum.attr import LayerIntegratedGradients, visualization


In [3]:
def pad_or_truncate(arr, target_len, pad_value=0):
    dtype = np.int16
    arr = np.asarray(arr, dtype = dtype)
    if len(arr) > target_len:
        # truncate
        return arr[:target_len]
    elif len(arr) < target_len:
        # padding
        pad_width = target_len - len(arr)
        return np.pad(arr, (0, pad_width), mode='constant', constant_values=dtype(pad_value))
    else:
        return arr
    
def forward_func(input_ids, model):
    outputs = model(input_ids=input_ids)
    return outputs.logits

In [ ]:
CURRUENT_PATH = Path.cwd()
OUTPUT_PATH = CURRUENT_PATH / 'outputs' /'train_text_classification_bert'

In [5]:
with open(f'/{CURRUENT_PATH}/imdb_preprocessed.pkl', 'rb') as f:
    train_x, train_y, valid_x, valid_y, test_x, test_y = pkl.load(f)


In [6]:
GPU_PARALLEL = True
model = torch.load(OUTPUT_PATH / 'best_model.pt', map_location='cpu')

device = torch.device('cuda' if torch.cuda.is_available else 'cpu')
model.to(device)
if GPU_PARALLEL:
    model = torch.nn.DataParallel(model)
model.eval()

DataParallel(
  (module): BertForSequenceClassification(
    (bert): BertModel(
      (embeddings): BertEmbeddings(
        (word_embeddings): Embedding(30522, 768, padding_idx=0)
        (position_embeddings): Embedding(512, 768)
        (token_type_embeddings): Embedding(2, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): BertEncoder(
        (layer): ModuleList(
          (0-11): 12 x BertLayer(
            (attention): BertAttention(
              (self): BertSelfAttention(
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Linear(in_features=768, out_features=768, bias=True)
                (value): Linear(in_features=768, out_features=768, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): BertSelfOutput(
                (dense): Linear(in_features=768, out_features=768, 

In [8]:
model.module.bert.embeddings if GPU_PARALLEL else model.bert.embeddings

BertEmbeddings(
  (word_embeddings): Embedding(30522, 768, padding_idx=0)
  (position_embeddings): Embedding(512, 768)
  (token_type_embeddings): Embedding(2, 768)
  (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
  (dropout): Dropout(p=0.1, inplace=False)
)

In [9]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")